In [ ]:
!pip install openai
import os

folders = [
    'references/cartoon',
    'references/watercolor',
    'references/flat',
    'cartoon',
    'watercolor',
    'flat'
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

In [3]:
import json
import base64
import os
import openai
import re
import time
import uuid
import statistics
from io import BytesIO
from PIL import Image

# ============================================================
# 1. API 与 路径配置
# ============================================================
client = openai.OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    timeout=240.0
)

MODEL_NAME = "google/gemini-2.0-flash-001"

STYLES_CONFIG = {"cartoon": "./cartoon", "watercolor": "./watercolor", "flat": "./flat"}
REFERENCES_DIR = "./references"
PROMPTS_FILE = "batch_sticker_prompts.json"
OUTPUT_FILE = "sticker_individual_scores.json"
REGEN_THRESHOLD = 7.0
# ============================================================
# 2. 图像加载
# ============================================================
def encode_high_quality_image(path):
    if not path or not os.path.exists(path):
        return None
    try:
        img = Image.open(path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=95)
        return base64.b64encode(buffer.getvalue()).decode('utf-8')
    except:
        return None

def load_style_references(style_name):
    ref_b64_list = []
    style_ref_dir = os.path.join(REFERENCES_DIR, style_name)
    if os.path.exists(style_ref_dir):
        fnames = [f for f in os.listdir(style_ref_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:2]
        for fname in fnames:
            b64 = encode_high_quality_image(os.path.join(style_ref_dir, fname))
            if b64:
                ref_b64_list.append(b64)
    return ref_b64_list

# ============================================================
# 3. 核心打分引擎
# ============================================================
def evaluate_sticker(gen_b64, ref_b64_list, raw_desc, style_name):
    session_id = str(uuid.uuid4())[:8]

    system_prompt = f"""
    Task ID: {session_id}
    You are a meticulous Art Quality Auditor. You must perform a side-by-side comparison between the TARGET STICKER and the provided metadata/references.

    SCORING DIMENSIONS (1.0-10.0 with 1 decimal precision):

    1. CONTENT ACCURACY (content_score, 50%):
       - Primary Goal: Compare the sticker with the "raw_image_description": "{raw_desc}".
       - Checklist:
         * Species/Subject: Is it the correct animal?
         * Pose & Action: Does the posture match the description?
         * Technical Note: The term 'sks' is a UNIQUE IDENTIFIER for the trained style. It is NOT a typo for 'ski'. Do not penalize if 'ski' equipment is missing; focus on the animal subject and its adherence to the 'sks' style. [cite: 13, 55, 61]

    2. STYLE CONSISTENCY (style_score, 40%):
       - Primary Goal: Evaluate if the sticker reflects the VISUAL DNA of the {style_name.upper()} Gold Standard references. [cite: 14, 32]
       - Checklist:
         * Line Work: Does it match the reference's line weight?
         * Texture & Shading: Does it mimic the reference's rendering?
         * Style feature: Is the consistent with the style samples?

    3. TECHNICAL QUALITY (quality_score, 10%):
       - Primary Goal: Professional sticker readiness.
       - Checklist:
         * Background: The background should be clear rather than chaotic.
         * Edges: Must be crisp.
         * AI Artifacts: There are no deformed limbs or parts that do not conform to reality.

    SCORING CALIBRATION RULES:
    - A score of 9.0+ means the sticker is excellent and has almost no visible flaws.
    - A score of 8.0+ means acceptable but clearly improvable.
    - A score of 7.0 or below should be given if there is any noticeable issue in subject accuracy, line quality, or visual cleanliness.

    HARD PENALTY RULES:
    - If the subject is not clearly correct, total score must not exceed 7.0.
    - If there are visible AI artifacts or distorted parts, total score must not exceed 7.0.
    - If style consistency is weak, total score must not exceed 7.0.

    MANDATORY RULES:
    - BE CRITICAL: Differentiate between 8.7 and 8.8 based on a specific flaw.
    - REASONING FIRST: You MUST provide a detailed "reasoning" BEFORE the scores. Describe specific visual evidence.
    - Output ONLY valid JSON.
    """

    user_content = [{"type": "text", "text": f"Reference styles for {style_name}:"}]
    for ref in ref_b64_list:
        user_content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{ref}"}})

    user_content.append({"type": "text", "text": "TARGET STICKER FOR AUDIT:"})
    user_content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{gen_b64}"}})

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )

        raw_res = response.choices[0].message.content
        data = json.loads(re.search(r'\{.*\}', raw_res, re.DOTALL).group(0))
        return {k.lower(): v for k, v in data.items()}
    except Exception:
        return None

def evaluate_sticker_three_times(gen_b64, ref_b64_list, raw_desc, style_name, max_attempts_per_round=3):
    """
    对同一张图进行 3 次独立打分，返回三次结果的中位数。
    如果某一轮失败，会在该轮内继续重试，直到拿到有效结果或达到最大尝试次数。
    """
    results = []

    for round_idx in range(3):
        round_res = None

        for attempt in range(max_attempts_per_round):
            round_res = evaluate_sticker(gen_b64, ref_b64_list, raw_desc, style_name)
            if round_res and "reasoning" in round_res:
                break
            time.sleep(1)

        if round_res and "reasoning" in round_res:
            try:
                c = float(round_res.get('content_score', 0))
                s = float(round_res.get('style_score', 0))
                q = float(round_res.get('quality_score', 0))
                reasoning = round_res.get('reasoning', "")
                results.append({
                    "content_score": c,
                    "style_score": s,
                    "quality_score": q,
                    "reasoning": reasoning
                })
            except Exception:
                continue

    if len(results) < 3:
        return None

    med_content = round(statistics.median([r["content_score"] for r in results]), 2)
    med_style = round(statistics.median([r["style_score"] for r in results]), 2)
    med_quality = round(statistics.median([r["quality_score"] for r in results]), 2)
    med_total = round(med_content * 0.5 + med_style * 0.4 + med_quality * 0.1, 2)

    return {
        "content_score": med_content,
        "style_score": med_style,
        "quality_score": med_quality,
        "sticker_average_score": med_total,
        "reasoning": [r["reasoning"] for r in results]
    }

# ============================================================
# 4. 主控管线
# ============================================================
def run_pipeline():
    print("[INFO] 启动独立高精度审计管线...")

    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            try:
                report = json.load(f)
            except:
                report = {}
    else:
        report = {}

    with open(PROMPTS_FILE, 'r', encoding='utf-8') as f:
        prompts_data = json.load(f)

    for style, folder in STYLES_CONFIG.items():
        if not os.path.exists(folder):
            continue

        print(f"\n" + "=" * 50)
        print(f">>> 正在审计风格: {style.upper()}")
        print("=" * 50)

        ref_images = load_style_references(style)
        if style not in report:
            report[style] = []
        done_ids = {res['file_id'] for res in report[style]}

        for item in prompts_data:
            base_name = item['file_name'].rsplit('.', 1)[0]
            save_id = None
            gen_path = None

            for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG']:
                test_path = os.path.join(folder, base_name + ext)
                if os.path.exists(test_path):
                    save_id = base_name + ext
                    gen_path = test_path
                    break

            if not save_id:
                continue

            if save_id in done_ids:
                continue

            if os.path.exists(gen_path):
                raw_desc = item.get('raw_image_description', item.get('sd_sticker_prompt', "A sticker"))
                g_b64 = encode_high_quality_image(gen_path)

                if g_b64:
                    print(f"  -> {save_id} 正在独立审计 ...", end="\r")

                    res = evaluate_sticker_three_times(g_b64, ref_images, raw_desc, style)

                    if res and "reasoning" in res:
                        try:
                            c = float(res.get('content_score', 0))
                            s = float(res.get('style_score', 0))
                            q = float(res.get('quality_score', 0))
                            reason = res.get('reasoning', [])

                            avg = float(res.get('sticker_average_score', 0))
                            status = "PASS" if avg >= REGEN_THRESHOLD else "REGENERATE"

                            report[style].append({
                                "file_id": save_id,
                                "metrics": {
                                    "content_score": c,
                                    "style_score": s,
                                    "quality_score": q,
                                    "reasoning": reason,
                                    "sticker_average_score": avg,
                                    "status": status
                                }
                            })

                            print(f"  -> {save_id} | [内容:{c} 风格:{s} 质量:{q}] | 总分:{avg} -> {status}")

                            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                                json.dump(report, f, indent=4, ensure_ascii=False)
                        except Exception:
                            print(f"  -> {save_id} | [ERR] 数据解析异常")
                    else:
                        print(f"  -> {save_id} | [FAIL] 无法获取审计理由或分数")

    print(f"\n[DONE] 审计任务结束。结果已保存至 {OUTPUT_FILE}")

if __name__ == "__main__":
    run_pipeline()

[INFO] 启动独立高精度审计管线...

>>> 正在审计风格: CARTOON
  -> animal_2.png | [内容:9.5 风格:9.0 质量:9.5] | 总分:9.3 -> PASS
  -> animal_26.png | [内容:9.5 风格:6.5 质量:7.5] | 总分:8.1 -> PASS
  -> animal_5.png | [内容:9.5 风格:8.5 质量:7.5] | 总分:8.9 -> PASS
  -> animal_6.png | [内容:9.0 风格:7.5 质量:8.5] | 总分:8.35 -> PASS
  -> animal_28.png | [内容:9.5 风格:8.5 质量:9.5] | 总分:9.1 -> PASS
  -> animal_29.png | [内容:7.5 风格:8.0 质量:7.8] | 总分:7.73 -> PASS
  -> animal_15.png | [内容:9.5 风格:8.8 质量:9.0] | 总分:9.17 -> PASS
  -> animal_36.png | [内容:9.5 风格:8.8 质量:9.5] | 总分:9.22 -> PASS
  -> animal_32.png | [内容:9.0 风格:8.5 质量:9.5] | 总分:8.85 -> PASS
  -> animal_11.png | [内容:9.5 风格:9.2 质量:9.7] | 总分:9.4 -> PASS
  -> animal_13.png | [内容:9.5 风格:8.5 质量:9.0] | 总分:9.05 -> PASS
  -> animal_27.png | [内容:9.5 风格:8.2 质量:9.0] | 总分:8.93 -> PASS
  -> animal_3.png | [内容:9.5 风格:6.5 质量:7.5] | 总分:8.1 -> PASS
  -> animal_7.png | [内容:9.0 风格:6.5 质量:7.5] | 总分:7.85 -> PASS
  -> animal_10.png | [内容:9.5 风格:8.2 质量:8.5] | 总分:8.88 -> PASS
  -> animal_9.png | [内容:9.0 风格:6.5 质量: